# Bölüm 6/7 — Arastirma Sorulari



## Önceki bölümden devam
Bu hücre, `5_...` bölümünde kaydedilen tüm değişkenleri ve tanımlı fonksiyonları geri yükler.

In [ ]:
!pip install dill --quiet
import dill
dill.load_session('checkpoint_5.pkl')
print('Önceki bölümün oturumu yüklendi.')

## Sorular

### 1. Dezenformasyon Hangi Konularda Yayılıyor?


Soru 1: Dezenformasyonun hangi haber kategorilerinde (news_type) yoğunlaştığı incelenir. Kategori büyüklükleri çok dengesiz olduğundan, yalnızca yeterli örneklemi olan (n≥30) kategoriler güvenilir kabul edilir ve iki panelli bir grafikte (hacim + oran) gösterilir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

kategori_ozet = df_clean.groupby("news_type")["is_disinformation"].agg(
    toplam="count",
    yes_orani=lambda x: (x == "yes").mean()
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: hacim (ilk 5 kategori)
ax1 = axes[0]
top5 = kategori_ozet.sort_values("toplam", ascending=False).head(5).sort_values("toplam")
ax1.barh(top5["news_type"], top5["toplam"], color="#4C72B0")
ax1.set_xlabel("Kayıt sayısı")
ax1.set_title("Kategorilere Göre Kayıt Hacmi (ilk 5)")
for i, (v, kat) in enumerate(zip(top5["toplam"], top5["news_type"])):
    ax1.text(v + 50, i, str(v), va="center", fontsize=9)

# Panel 2: yes orani, sadece n>=30 kategoriler
ax2 = axes[1]
guvenilir = kategori_ozet[kategori_ozet["toplam"] >= 30].sort_values("yes_orani")
genel_ortalama = (df_clean["is_disinformation"] == "yes").mean()
colors = ["#c1440e" if o > genel_ortalama else "#4C72B0" for o in guvenilir["yes_orani"]]
bars = ax2.barh(guvenilir["news_type"], guvenilir["yes_orani"], color=colors)
ax2.axvline(genel_ortalama, color="gray", linestyle="--", linewidth=1,
            label=f"Genel ortalama ({genel_ortalama:.3f})")
ax2.set_xlabel("'yes' (dezenformasyon) oranı")
ax2.set_title("Kategori Bazında Dezenformasyon Oranı\n(sadece n≥30 kategoriler)")
ax2.legend()
for bar, n in zip(bars, guvenilir["toplam"]):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f"n={n}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

### 2. Dezenformasyon içeren metinler ile içermeyen metinlerin uzunlukları arasında bir fark var mı?


Soru 2: Dezenformasyon içeren ve içermeyen metinlerin uzunlukları karşılaştırılır (boxplot ve histogram). Bu analiz, projedeki en çarpıcı bulgulardan birini ortaya çıkarır: dezenformasyon metinleri gerçek haberlerden ortalama 4 kat daha uzundur.

In [ ]:
import matplotlib.pyplot as plt

df_clean["uzunluk"] = df_clean["clean_text"].str.len()

no_uz = df_clean[df_clean["is_disinformation"] == "no"]["uzunluk"]
yes_uz = df_clean[df_clean["is_disinformation"] == "yes"]["uzunluk"]

print("no  ort. uzunluk:", no_uz.mean().round(1), "| medyan:", no_uz.median())
print("yes ort. uzunluk:", yes_uz.mean().round(1), "| medyan:", yes_uz.median())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
bp = ax1.boxplot([no_uz, yes_uz], tick_labels=["no (gerçek haber)", "yes (dezenformasyon)"],
                  patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], ["#4C72B0", "#c1440e"]):
    patch.set_facecolor(color)
ax1.set_ylabel("Metin uzunluğu (karakter)")
ax1.set_title("Metin Uzunluğu Dağılımı (Boxplot)")

ax2 = axes[1]
ax2.hist(no_uz, bins=40, alpha=0.6, label="no (gerçek haber)", color="#4C72B0")
ax2.hist(yes_uz, bins=40, alpha=0.6, label="yes (dezenformasyon)", color="#c1440e")
ax2.set_xlabel("Metin uzunluğu (karakter)")
ax2.set_ylabel("Kayıt sayısı")
ax2.set_title("Metin Uzunluğu Histogramı")
ax2.legend()

plt.tight_layout()
plt.show()

### 3. Dezenformasyon içeren metinlerde en sık hangi kelimeler kullanılıyor?


Soru 3: Dezenformasyon içeren metinlerde en sık geçen kelimeler `CountVectorizer` ile tespit edilir. Ham sayımın yanı sıra, sadece dezenformasyon metinlerinde geçip gerçek haberde hiç görülmeyen kelimeler de ayrıca listelenir — bu liste, model katsayı analizindeki bulgularla bağımsız olarak örtüşmektedir.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

yes_metinler = df_clean[df_clean["is_disinformation"] == "yes"]["clean_text"]
no_metinler  = df_clean[df_clean["is_disinformation"] == "no"]["clean_text"]

cv_yes = CountVectorizer(stop_words="english", min_df=5)
X_yes = cv_yes.fit_transform(yes_metinler)
freq_yes_ham = pd.Series(np.asarray(X_yes.sum(axis=0)).ravel(), index=cv_yes.get_feature_names_out())
freq_yes_oran = freq_yes_ham / freq_yes_ham.sum()

cv_no = CountVectorizer(stop_words="english", min_df=5)
X_no = cv_no.fit_transform(no_metinler)
freq_no_oran = pd.Series(np.asarray(X_no.sum(axis=0)).ravel(), index=cv_no.get_feature_names_out())
freq_no_oran = freq_no_oran / freq_no_oran.sum()

# sadece "yes" metinlerinde gecen (no'da hic gormedigimiz) kelimeler
sadece_yes = freq_yes_oran.index.difference(freq_no_oran.index)
top_sadece_yes = freq_yes_oran[sadece_yes].sort_values(ascending=False).head(12)
top_genel = freq_yes_ham.sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

ax1 = axes[0]
ax1.barh(top_genel.index[::-1], top_genel.values[::-1], color="#4C72B0")
ax1.set_title("Dezenformasyon Metinlerinde\nEn Sık Geçen Kelimeler")
ax1.set_xlabel("Toplam geçiş sayısı")

ax2 = axes[1]
ax2.barh(top_sadece_yes.index[::-1], top_sadece_yes.values[::-1], color="#c1440e")
ax2.set_title("Sadece Dezenformasyonda Geçen\n(Gerçek Haberde Hiç Görülmeyen) Kelimeler")
ax2.set_xlabel("Oransal frekans")

plt.tight_layout()
plt.show()

### Hipotez — Dezenformasyon ve gerçek haberler farklı kelime dünyalarından besleniyor

Ek bir görsel olarak, dezenformasyon ve gerçek haber metinlerinin kelime bulutları (`WordCloud`) karşılaştırmalı şekilde oluşturulur. Bu, Soru 3'teki çubuk grafiklerin alternatif/tamamlayıcı bir sunumudur.

In [ ]:
!pip install wordcloud --quiet
from wordcloud import WordCloud, STOPWORDS

fig, axes = plt.subplots(1, 2, figsize=(14,6))
for ax, label in zip(axes, ["yes", "no"]):
    text = " ".join(df_clean[df_clean["is_disinformation"]==label]["clean_text"])
    wc = WordCloud(width=600, height=400, background_color="white",
                    stopwords=STOPWORDS).generate(text)
    ax.imshow(wc); ax.axis("off")
    ax.set_title(f"is_disinformation = {label}")
plt.show()

### 4. Yalan haberler ile doğru haberlerin duygu tonları arasında anlamlı bir fark var mı?


Soru 4'ün duygu analizi için gereken `nltk` kütüphanesinin VADER duygu sözlüğü indirilir.

In [ ]:
import nltk
nltk.download("vader_lexicon")

Soru 4: `VADER` duygu analiz aracı ile her metnin duygu skoru (-1 ile +1 arası) hesaplanır. Dezenformasyon ve gerçek haber metinlerinin duygu tonları istatistiksel olarak (Mann-Whitney U testi) karşılaştırılır ve boxplot ile görselleştirilir.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

sia = SentimentIntensityAnalyzer()

# compound skor: -1 (tamamen negatif) ile +1 (tamamen pozitif) arasi tek bir ozet skor
df_clean["sentiment"] = df_clean["clean_text"].apply(lambda t: sia.polarity_scores(t)["compound"])

yes_sent = df_clean[df_clean["is_disinformation"] == "yes"]["sentiment"]
no_sent  = df_clean[df_clean["is_disinformation"] == "no"]["sentiment"]

print("yes ort. duygu skoru:", yes_sent.mean().round(4), "| medyan:", yes_sent.median())
print("no  ort. duygu skoru:", no_sent.mean().round(4), "| medyan:", no_sent.median())

stat, p_value = mannwhitneyu(yes_sent, no_sent)
print("Mann-Whitney U p-değeri:", p_value)

fig, ax = plt.subplots(figsize=(7,5))
bp = ax.boxplot([no_sent, yes_sent], tick_labels=["no (gerçek haber)","yes (dezenformasyon)"],
                 patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], ["#4C72B0","#c1440e"]):
    patch.set_facecolor(color)
ax.set_ylabel("VADER Duygu Skoru (-1: negatif, +1: pozitif)")
ax.set_title("Dezenformasyon vs Gerçek Haber - Duygu Tonu Karşılaştırması")
plt.tight_layout()
plt.show()

### 5. Tekli kelimelerden (unigram) ziyade, birbiriyle sık kullanılan ikili (bigram) ve üçlü (trigram) kelime gruplarını analiz etmek.


Soru 5: Tekil kelimeler yerine, dezenformasyon metinlerinde sık birlikte geçen ikili (bigram) ve üçlü (trigram) kelime grupları `CountVectorizer(ngram_range=(2,2)/(3,3))` ile tespit edilip görselleştirilir.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

yes_metinler = df_clean[df_clean["is_disinformation"] == "yes"]["clean_text"]

cv_bi = CountVectorizer(stop_words="english", ngram_range=(2,2), min_df=5)
X_bi = cv_bi.fit_transform(yes_metinler)
freq_bi = pd.Series(np.asarray(X_bi.sum(axis=0)).ravel(),
                     index=cv_bi.get_feature_names_out()).sort_values(ascending=False).head(12)

cv_tri = CountVectorizer(stop_words="english", ngram_range=(3,3), min_df=5)
X_tri = cv_tri.fit_transform(yes_metinler)
freq_tri = pd.Series(np.asarray(X_tri.sum(axis=0)).ravel(),
                      index=cv_tri.get_feature_names_out()).sort_values(ascending=False).head(12)

sablon_isaretleri = ["fakes", "narratives", "propaganda", "digest"]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

ax1 = axes[0]
ax1.barh(freq_bi.index[::-1], freq_bi.values[::-1], color="#4C72B0")
ax1.set_title("Dezenformasyon Metinlerinde\nEn Sık Geçen Bigramlar (2'li kelime grubu)")
ax1.set_xlabel("Geçiş sayısı")

ax2 = axes[1]
colors = ["#c1440e" if any(s in k for s in sablon_isaretleri) else "#4C72B0" for k in freq_tri.index]
ax2.barh(freq_tri.index[::-1], freq_tri.values[::-1], color=colors[::-1])
ax2.set_title("Dezenformasyon Metinlerinde\nEn Sık Geçen Trigramlar (3'lü kelime grubu)")
ax2.set_xlabel("Geçiş sayısı")

plt.tight_layout()
plt.show()

### 6. Dezenformasyonu tespit etmede makine öğrenmesi modelleri ne kadar başarılı?


Soru 6: Optimize edilmiş altı modelin tamamının genel başarısı — accuracy ve "no" sınıfı recall açısından — tek bir özet grafikte karşılaştırılarak makine öğrenmesinin dezenformasyon tespitindeki genel etkinliği değerlendirilir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

model_tahminleri = {
    "Multinomial\nNB (optimize)":         (y_test, y_pred_best_mnb),
    "Complement\nNB (optimize)":          (y_test, y_pred_best_cnb),
    "Logistic\nRegression (optimize)":    (y_test, y_pred_best_lr),
    "Linear SVM\n(optimize)":  (y_test, y_pred_best_svm),
    "Random Forest\n(optimize)": (y_test, y_pred_best_rf),
    "XGBoost\n(optimize)":     (y_test_xgb, y_pred_best_xgb),
}

modeller, accuracy, no_recall = [], [], []
for isim, (yt, yp) in model_tahminleri.items():
    target_names = ["no", "yes"] if set(np.unique(yt)) == {0, 1} else None
    rapor = classification_report(yt, yp, target_names=target_names, output_dict=True)
    modeller.append(isim)
    accuracy.append(accuracy_score(yt, yp))
    no_recall.append(rapor["no"]["recall"])

baseline = (y_test == "yes").mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(modeller)))

ax1 = axes[0]
bars1 = ax1.bar(modeller, accuracy, color=colors)
ax1.axhline(baseline, color="red", linestyle="--", linewidth=1, label=f"Baseline ({baseline:.3f})")
ax1.set_ylim(0.7, 1.0)
ax1.set_ylabel("Accuracy")
ax1.set_title("Modellerin Dezenformasyon Tespitindeki\nGenel Başarısı (Accuracy)")
ax1.legend()
for bar, v in zip(bars1, accuracy):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)

ax2 = axes[1]
bars2 = ax2.bar(modeller, no_recall, color=colors)
ax2.set_ylim(0.4, 1.02)
ax2.set_ylabel("'no' (gerçek haber) Recall")
ax2.set_title("Modellerin Gerçek Haberi Yanlış\nEtiketlememe Başarısı")
for bar, v in zip(bars2, no_recall):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

### 7.  NLP modellerinin hangi kategorilerdeki dezenformasyonu saptamada daha çok zorlandığını belirlemek.


Soru 7: Optimize edilmiş altı modelin (Linear SVM, Complement NB, Random Forest, Logistic Regression, Multinomial NB, XGBoost) haber kategorisi bazında dezenformasyon yakalama ("yes" recall) başarıları karşılaştırılır. Not: Bir modelin test setindeki genel "no" recall değeri düşükse (yani modelin "yes"e önyargılı davrandığı anlaşılıyorsa), bu artık sabit bir liste yerine ölçülen değere göre otomatik olarak işaretlenir; çünkü optimizasyon sonrası hangi modelin önyargılı kalacağı önceden bilinemez.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

modeller = {
    "Logistic Regression (optimize)": best_lr,
    "Multinomial NB (optimize)": best_mnb,
    "Linear SVM (optimize)": best_svm,
    "Complement NB (optimize)": best_cnb,
    "Random Forest (optimize)": best_rf,
    "XGBoost (optimize)": best_xgb,
}

def tahmin_al(model):
    """XGBoost 0/1 tahmin dondurur, digerleri 'no'/'yes' -- burada hepsini 'no'/'yes'e normalize ediyoruz."""
    pred = model.predict(X_test_tfidf)
    if set(np.unique(pred)) <= {0, 1}:
        return np.where(pred == 1, "yes", "no")
    return pred

# Onyargi artik sabit bir liste degil, olculen "no" recall degerine gore otomatik belirleniyor
NO_RECALL_ESIK = 0.85
onyargili = set()
for isim, model in modeller.items():
    pred = tahmin_al(model)
    rapor = classification_report(y_test, pred, output_dict=True, zero_division=0)
    if rapor["no"]["recall"] < NO_RECALL_ESIK:
        onyargili.add(isim)
print("Bu esikte onyargili (\"no\" recall < %.2f) bulunan modeller:" % NO_RECALL_ESIK, onyargili or "yok")

test_df_base = df_clean.loc[X_test_text.index].copy()
test_df_base["gercek"] = y_test.values

sonuclar = {}
for isim, model in modeller.items():
    pred = tahmin_al(model)
    tdf = test_df_base.copy()
    tdf["tahmin"] = pred
    tdf["dogru_mu"] = tdf["gercek"] == tdf["tahmin"]
    yes_df = tdf[tdf["gercek"] == "yes"]
    rec = yes_df.groupby("news_type")["dogru_mu"].mean()
    n = yes_df.groupby("news_type")["dogru_mu"].count()
    sonuclar[isim] = rec[n >= 30]

kategoriler = sorted(set().union(*[s.index for s in sonuclar.values()]),
                      key=lambda k: -sum(s.get(k,0) for s in sonuclar.values()))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(kategoriler))
n_model = len(sonuclar)
width = 0.8 / n_model

for i, (isim, seri) in enumerate(sonuclar.items()):
    degerler = [seri.get(k, 0) for k in kategoriler]
    offset = (i - n_model/2) * width + width/2
    hatch = "///" if isim in onyargili else None
    renk = "#b0b0b0" if isim in onyargili else plt.cm.tab10(i)
    ax.bar(x + offset, degerler, width, label=isim + (" *" if isim in onyargili else ""),
           color=renk, hatch=hatch, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(kategoriler)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel("Dezenformasyon Yakalama Oranı ('yes' recall)")
ax.set_title("Kategori Bazında Model Karşılaştırması\n(* işaretli modeller 'yes'e önyargılı — bu recall değerleri yanıltıcı olabilir)")
ax.legend(fontsize=8, loc="lower left")

plt.tight_layout()
plt.show()


### 8.  Veri setinde yer alan etiket güvenilirlik (confidence) değerlerinin, modelin öğrenme süreciyle nasıl bir ilişkisi olduğunu incelemek.

Soru 8: Veri setindeki etiket güven (`confidence`) değeri ile en iyi modelin (yukarıda otomatik seçilen) test performansı arasındaki ilişki incelenir. Verinin bu sütunda çok az çeşitlilik göstermesi (kayıtların %93'ü aynı değerde) nedeniyle, bulgular temkinli yorumlanmalıdır.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

test_df = df_clean.loc[X_test_text.index].copy()
test_df["gercek"] = y_test.values
# Sabit bir model yerine, sonuc_tablosu'ndan otomatik secilen en iyi model (en_iyi_model) kullanilir
_ham_tahmin = en_iyi_model.predict(X_test_tfidf)
test_df["tahmin"] = np.where(np.isin(_ham_tahmin, [0, 1]), np.where(_ham_tahmin == 1, "yes", "no"), _ham_tahmin)
test_df["dogru_mu"] = test_df["gercek"] == test_df["tahmin"]

conf_ozet = test_df.groupby("confidence")["dogru_mu"].agg(dogruluk="mean", n="count")
print(conf_ozet)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
bars = ax1.bar(conf_ozet.index.astype(str), conf_ozet["n"], color="#4C72B0")
ax1.set_ylabel("Test setindeki kayıt sayısı")
ax1.set_title("Confidence Değerine Göre\nVeri Dağılımı (çok dengesiz)")
for bar, v in zip(bars, conf_ozet["n"]):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 20, str(v), ha="center", fontsize=9)

ax2 = axes[1]
colors = ["#4C72B0" if n >= 50 else "#c1c1c1" for n in conf_ozet["n"]]
bars2 = ax2.bar(conf_ozet.index.astype(str), conf_ozet["dogruluk"], color=colors)
ax2.set_ylim(0.85, 1.02)
ax2.set_ylabel("Model Doğruluğu")
ax2.set_title("Confidence Değerine Göre\nModel Doğruluğu (n<50 gri = güvenilmez)")
for bar, v, n in zip(bars2, conf_ozet["dogruluk"], conf_ozet["n"]):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}\n(n={n})", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

## Bu bölümü kaydet
Bir sonraki bölümün bu noktadan devam edebilmesi için tüm oturum (değişkenler, modeller, fonksiyonlar) diske kaydedilir.

In [ ]:
import dill
dill.dump_session('checkpoint_6.pkl')
print('Oturum checkpoint_6.pkl olarak kaydedildi.')